In [1]:
import os
import csv
import json
import oracledb
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_oracledb.vectorstores.oraclevs import OracleVS
from langchain_community.vectorstores.utils import DistanceStrategy
from langchain_oracledb.retrievers.text_search import create_text_index, OracleTextSearchRetriever

d:\IE103_Final_Project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
username = "system"
password = "oracle"
dsn = "192.168.1.248:1521/FREEPDB1"

In [3]:
try:
    connection = oracledb.connect(user=username, password=password, dsn=dsn)
    print("Connection successful!")

except oracledb.Error as e:
    error_obj, = e.args
    print(f"Oracle Error: {error_obj.message}")

except Exception as e:
    print(f"Undefined Error: {e}")

Connection successful!


In [4]:
corpus_path = os.path.join(os.path.dirname(os.getcwd()), "scifact", "corpus.jsonl")
corpus_path

'd:\\IE103_Final_Project\\scifact\\corpus.jsonl'

In [5]:
documents_langchain = []

with open(corpus_path, "r", encoding="utf-8") as document_jsonl_list:
    for line in document_jsonl_list:
        doc = json.loads(line)
        metadata = {"id": doc["_id"], "title": doc["title"]}
        doc_langchain = Document(page_content=doc["text"], metadata=metadata)
        documents_langchain.append(doc_langchain)

In [6]:
model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    encode_kwargs={"normalize_embeddings": True}
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3810.92it/s]


In [7]:
vector_store = OracleVS(
    client=connection,
    embedding_function=model,
    table_name="SciFact",
    distance_strategy=DistanceStrategy.COSINE,
)

In [8]:
create_text_index(
    client=connection,
    idx_name="keyword_scifact",
    vector_store=vector_store,
)

In [9]:
def RetrieveTopK(vector_store: OracleVS, k: int) -> OracleTextSearchRetriever:
    retriever = OracleTextSearchRetriever(
        vector_store=vector_store,
        k=k,
        fuzzy=True
    )
    return retriever

In [10]:
retriever = RetrieveTopK(vector_store, 10)

In [11]:
retrieval_results = retriever.invoke("0-dimensional biomaterials show inductive properties.")
retrieval_results

[Document(id='20b78b29-3e19-4f8a-81c5-154932ce341f', metadata={'id': '20b78b29-3e19-4f8a-81c5-154932ce341f', 'title': 'Topical application of chlorhexidine to neonatal umbilical cords for prevention of omphalitis and neonatal mortality in a rural district of Pakistan: a community-based, cluster-randomised trial.', 'corpus_id': '37118634'}, page_content='number NCT00682006. FINDINGS 187 clusters were randomly allocated to one of the four study groups. Of 9741 newborn babies delivered by participating TBAs, factorial analysis indicated a reduction in risk of omphalitis with CHX application (risk ratio [RR]=0·58, 95% CI 0·41-0·82; p=0·002) but no evidence of an effect of handwashing (RR=0·83, 0·61-1·13; p=0·24). We recorded strong evidence of a reduction in neonatal mortality in neonates who received CHX cleansing (RR=0·62, 95 % CI 0·45-0·85;'),
 Document(id='e997b023-8fed-4498-8b7c-378bc1b6b6a3', metadata={'id': 'e997b023-8fed-4498-8b7c-378bc1b6b6a3', 'title': 'An atlas of active enhance

In [12]:
def ChunksToDocuments(retrieval_results: list[Document], documents_langchain: list[Document]) -> list[Document]:
    corpus_results = set()
    for result in retrieval_results:
        corpus_id = result.metadata["corpus_id"]
        corpus_results.add(corpus_id)

    document_results = []
    for document in documents_langchain:
        corpus_id = document.metadata["id"]
        if corpus_id in corpus_results:
            document_results.append(document)

    return document_results

In [13]:
document_results = ChunksToDocuments(retrieval_results, documents_langchain)
document_results

[Document(metadata={'id': '4465608', 'title': 'An atlas of active enhancers across human cell types and tissues'}, page_content='Enhancers control the correct temporal and cell-type-specific activation of gene expression in multicellular eukaryotes. Knowing their properties, regulatory activity and targets is crucial to understand the regulation of differentiation and homeostasis. Here we use the FANTOM5 panel of samples, covering the majority of human tissues and cell types, to produce an atlas of active, in vivo-transcribed enhancers. We show that enhancers share properties with CpG-poor messenger RNA promoters but produce bidirectional, exosome-sensitive, relatively short unspliced RNAs, the generation of which is strongly related to enhancer activity. The atlas is used to compare regulatory programs between different cells at unprecedented depth, to identify disease-associated regulatory single nucleotide polymorphisms, and to classify cell-type-specific and ubiquitous enhancers. W

In [14]:
def DocumentsToCorpusID(document_results: list[Document]) -> list[str]:
    corpus_ids = []
    for document in document_results:
        corpus_id = document.metadata["id"]
        corpus_ids.append(corpus_id)
    
    return corpus_ids

In [15]:
corpus_ids = DocumentsToCorpusID(document_results)
corpus_ids

['4465608',
 '4702639',
 '6504953',
 '6550579',
 '7581911',
 '8891333',
 '10608397',
 '10906636',
 '12824568',
 '37118634']

In [16]:
test_path = os.path.join(os.path.dirname(os.getcwd()), "scifact", "qrels", "test.tsv")
test_path

'd:\\IE103_Final_Project\\scifact\\qrels\\test.tsv'

In [17]:
test = {}

with open(test_path, "r", encoding="utf-8") as f:
    reader = csv.DictReader(f, delimiter='\t')

    for row in reader:
        query_id = row["query-id"]
        corpus_id = row["corpus-id"]
        test[query_id] = corpus_id

In [18]:
for query_id, corpus_id in test.items():
    print(f"Query ID: {query_id}. Corpus ID: {corpus_id}.")
    break

Query ID: 1. Corpus ID: 31715818.


In [19]:
queries_path = os.path.join(os.path.dirname(os.getcwd()), "scifact", "queries.jsonl")
queries_path

'd:\\IE103_Final_Project\\scifact\\queries.jsonl'

In [20]:
queries = {}

with open(queries_path, "r", encoding="utf-8") as document_jsonl_list:
    for line in document_jsonl_list:
        doc = json.loads(line)
        query_id = doc["_id"]
        text = doc["text"]
        queries[query_id] = text

In [21]:
for query_id, query_text in queries.items():
    print(f"Query ID: {query_id}. Query text: {query_text}")
    break

Query ID: 0. Query text: 0-dimensional biomaterials lack inductive properties.


In [22]:
for k in range(5, 51, 5):
    correct = 0
    retriever = RetrieveTopK(vector_store, k)

    for i, (query_id, corpus_id) in enumerate(test.items()):
        retrieval_results = retriever.invoke(queries[query_id])
        document_results = ChunksToDocuments(retrieval_results, documents_langchain)
        corpus_ids = DocumentsToCorpusID(document_results)

        if corpus_id in corpus_ids:
            correct += 1

    print(f"Hit@{k}: {correct / 300 * 100:.2f}%")

Hit@5: 59.67%
Hit@10: 65.67%
Hit@15: 68.67%
Hit@20: 70.33%
Hit@25: 72.00%
Hit@30: 73.67%
Hit@35: 74.67%
Hit@40: 76.33%
Hit@45: 78.00%
Hit@50: 78.67%
